# Part A Report — Location Matching (Entity Resolution)

This report documents the methodology and results for **Part A** 

**Goal:** Link each record in `raw_financials` to the correct business location in `business_locations`.  
**Deliverables produced:** a reproducible matching pipeline + a financial table containing matched location IDs.

## Deliverables (Part A)

1. **Match table** (`outputs/final_matches.parquet`)
   - One row per `raw_id`
   - Best predicted `location_id` plus confidence metadata

2. **Financial table (single-file deliverable)**  
   - `outputs/financial_table.parquet`  
   - `outputs/financial_table.csv`  
   Includes both:
   - analysis-ready columns (matched location IDs + revenue window metrics)
   - diagnostic columns (match scores, match_source, etc.) in the same file for transparency

## Data

### raw_financials
Columns:
- `id`, `name`, `address`, `city`, `state`, `postal_code`, `start_at`, `end_at`, `revenue`

Interpretation:
- Each row is a financial record for a business over a date window (`start_at` to `end_at`).
- `revenue` is assumed to represent **card transactions only** (Part B will estimate total revenue).

### business_locations
Columns:
- `id`, `business_entity_id`, `name`, `street_address`, `city`, `state`, `postal_code`, `area_sq_ft`

Interpretation:
- Each row represents a location/POI.  
- Known to contain real-world inconsistencies: name variations, address typos, formatting differences.

## Matching Strategy (Overview)

The pipeline follows a standard entity-resolution pattern:

1) **Standardize text fields** (name/address/city/state/zip)  
2) **Blocking / candidate generation** to avoid all-to-all comparisons  
3) **Fuzzy similarity scoring** on normalized strings (name + address)  
4) **Best-match selection** per `raw_id`, with a confidence label:
   - `match` (high confidence)
   - `possible` (plausible but ambiguous)
   - `no_match` (candidates exist but similarity too low)
   - `unmatched` (no candidate found in either pass)

## Step 1 — Standardization (Normalization)

Before matching, I normalize both datasets to reduce formatting noise so fuzzy similarity reflects true similarity.

### Business name normalization (raw.name, locations.name)
- lowercase
- ASCII folding (remove accents/diacritics)
- remove punctuation; collapse whitespace
- normalize `& → and`
- remove legal suffix tokens (e.g., `inc`, `llc`, `corp`, `ltd`)

**Rationale:** These differences are common across systems and usually do not indicate different businesses.

### Address normalization (raw.address, locations.street_address)
- lowercase
- ASCII folding (remove accents/diacritics)
- remove punctuation; collapse whitespace
- remove unit/suite fragments (`apt`, `unit`, `ste`, `suite`, `#...`)
- normalize common street terms (e.g., `street → st`, `road → rd`, `avenue → ave`, `boulevard → blvd`, etc.)

**Rationale:** Addresses are highly discriminative but frequently differ in formatting.

## Step 2 — Blocking (Candidate Generation)

Comparing every raw record to every location is infeasible, so I use blocking to generate plausible candidates.

### Pass 1 (Primary block): `state + zip5`
- Candidate pairs are created only when `(state, postal_code)` matches exactly.
- This provides high precision and small candidate sets per record.

### Pass 2 (Fallback block): `state + city + zip3` with Top-K pruning
Some records have **no candidates** under the strict zip5 block (e.g., zips present in raw but absent in POI data, or other inconsistencies).
A broader `state+city` block can explode in large cities, so I use a tighter fallback:
- block on `(state, city, zip3)`
- keep only **Top-K** candidates per raw record using cheap heuristics (street number match, length similarity)
- then apply full fuzzy scoring on this reduced candidate set

**Rationale:** Improves recall while controlling compute and memory usage.

## Step 3 — Similarity Scoring

For each candidate pair, compute fuzzy similarity scores using token-based matching:

- `name_score` between normalized names
- `addr_score` between normalized addresses

Final score:
- `final_score = 0.35 * name_score + 0.65 * addr_score`

**Why address is weighted higher:**  
Chain names repeat across locations; street address is more location-specific and reduces false matches.

## Step 4 — Match Decision (Best Candidate + Confidence Label)

For each `raw_id`, select the candidate with the highest `final_score`.

To handle ambiguity:
- `second_score` is the runner-up candidate score (NULL if no runner-up exists)
- `gap = final_score - second_score` (NULL if no runner-up exists)

Decision rules:
- **match** if `final_score ≥ 90` and (no runner-up OR `gap ≥ 3`)
- **possible** if `final_score ≥ 85` but not strong enough for match
- **no_match** otherwise
- **unmatched** if no candidate exists (not present in final_matches; appears as unmatched in financial_table)

Interpretation:
- `match`: high confidence linkage
- `possible`: plausible but ambiguous (kept for transparency)
- `no_match`: candidate exists but similarity is insufficient
- `unmatched`: no candidate found in either pass

## Why `second_score` and `gap` are often NULL

After blocking (especially zip5 blocking), most `raw_id`s have only **one** candidate location.
In those cases:
- there is no runner-up candidate
- `second_score` is NULL
- `gap` is NULL

This is expected and generally indicates low ambiguity because the candidate set is already highly constrained by geography.

## Key Assumptions

1. Geographic fields (`state`, `postal_code`) are reliable enough to use as the primary blocking key for most records.
2. Address provides stronger location identity than name; therefore it receives higher weight in the final score.
3. Thresholds (`90/85`) are set to favor precision while keeping a “possible” bucket for ambiguous cases.
4. The fallback block `(state, city, zip3)` recovers records missing zip5 candidates while keeping candidate sets bounded by Top-K pruning.

## Limitations & Tradeoffs

1. **No ground-truth labels:** Match quality is heuristic-based (fuzzy similarity + sanity checks).  
2. **False negatives possible:** If a record has incorrect geographic fields, the correct POI may not appear in either blocking pass.  
3. **Top-K pruning risk (fallback):** Rarely, the correct match might be pruned if it does not rank in the Top-K heuristic shortlist.

## Transition to Part B

The output of Part A provides:
- `matched_location_id` to join revenue to POI features (e.g., `business_entity_id`, `area_sq_ft`)
- time-window metrics (`days_in_window`, `card_rev_per_day`)
- confidence metadata (`match_status`, `match_source`, `final_score`)

Part B will build a revenue adjustment (multiplier) system to estimate total revenue from card revenue, and create comparative visualizations and theoretical validation answers.